# ⚡ SupremeAI — Brain Distillation & Instant Patch Cache Forge
**Objective:** Run LLMs (DeepSeek / Qwen Coder) on Kaggle GPU to pre-compute error fixes, AST refactoring patterns, and boilerplate patches to store in Cloudflare KV / Supabase cache.

In [ ]:
# 1. Environment & Dependencies
!nvidia-smi
!pip install -q transformers accelerate bitsandbytes torch loguru

In [ ]:
import os
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from loguru import logger

model_id = 'Qwen/Qwen2.5-Coder-1.5B-Instruct'
logger.info(f'Loading model: {model_id}...')
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, device_map='auto')
logger.success('Model loaded into GPU memory.')

In [ ]:
# 2. Pre-computed Patch Generation Loop
common_patterns = [
    {'id': 'fastapi_cors_setup', 'prompt': 'Write optimal production FastAPI CORS middleware configuration'},
    {'id': 'cloudflare_turnstile_verify', 'prompt': 'Write Python async siteverify function for Cloudflare Turnstile token'},
    {'id': 'supabase_pgvector_query', 'prompt': 'Write Python SQLAlchemy query to match vectors using cosine similarity'}
]

cached_patches = {}
for item in common_patterns:
    logger.info(f'Synthesizing patch for: {item["id"]}...')
    inputs = tokenizer(item['prompt'], return_tensors='pt').to('cuda')
    outputs = model.generate(**inputs, max_new_tokens=512, temperature=0.2)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    cached_patches[item['id']] = response

with open('/tmp/distilled_patch_cache.json', 'w') as f:
    json.dump(cached_patches, f, indent=2)
logger.success(f'Saved {len(cached_patches)} distilled patches into cache artifact.')